# 采样性能对比

我们唯一称得上创新点的小玩意，虽然在整个推理流程中比例不大，但理论上应该是快了一些

## 1. 初始化

加载模型，完成prefill阶段，得到初始logits

In [1]:
from helper import load_from_cache, prefill

base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

prompt = "python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
logits, _ = prefill(model, params, input_ids)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


## 2. 定义采样函数

- 基线：什么都不干，输入logits，返回logits中的第一个，作为基础通信量和时间的参考
- 贪心采样：取原始值最高的logit输出，无需softmax等计算，V-1次比较
- topk采样：取原始值最高的k个，在这k个中取样，O(kV)
- minp采样：找到最高概率pmax，设置最低概率pmin=minp*pmax，在概率pmin-pmax的logits中采样
- 老代码：跑出来非常糟糕结果的老采样代码

In [2]:
import jax
import jax.numpy as jnp

# 基线，只测时间
def sampler_baseline(logits):
    return [0]

# 贪心采样
def sampler_greedy(logits):
    return jnp.argmax(logits, axis=-1)

# top k
def sampler_top_k(logits, top_k = 40, seed = 42):

    key = jax.random.PRNGKey(seed)

    # 1. 找到第 k 大的 logit 值（阈值）
    values, _ = jax.lax.top_k(logits, k=top_k)
    kth = values[:, -1:]                     # 形状 (batch, 1)

    # 2. 掩码：小于阈值的设为 -1e9（或 -inf）
    masked_logits = jnp.where(logits < kth, -1e9, logits)

    # 3. 直接采样（categorical 内部会做 softmax）
    return jax.random.categorical(key, masked_logits, axis=-1)

# min p，变形优化版本
def sampler_min_p(logits, min_p =0.1, seed = 42):

    key = jax.random.PRNGKey(seed)

    max_logit = jnp.max(logits, axis=-1, keepdims=True)
    threshold = jnp.log(min_p) + max_logit # math hack
    mask = logits >= threshold
    filtered_logits = jnp.where(mask, logits, -1e10)

    # 随机采样
    key, subkey = jax.random.split(key)
    return jax.random.categorical(subkey, filtered_logits, axis=-1)

# 老代码
def sampler_legacy(logits, sample: bool = True, top_k: int = 40, seed: int = 42):
    key = jax.random.PRNGKey(seed)
    if top_k is not None:
        values, _ = jax.lax.top_k(logits, k=top_k)
        kth_values = values[:, -1:]
        logits = jnp.where(logits < kth_values, -1e9, logits)

    probs = jax.nn.softmax(logits, axis=-1)

    if sample:
        key, subkey = jax.random.split(key)
        next_id = jax.random.categorical(subkey, jnp.log(probs + 1e-9), axis=-1)
    else:
        next_id = jnp.argmax(probs, axis=-1)

    return next_id


## 3. 明文测试

In [3]:
next_id = sampler_baseline(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_top_k(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_min_p(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_legacy(logits)
print(prompt, tokenizer.decode(next_id), sep="")

python is<|endoftext|>
python is not
python is in
python is in


## 4. spu测试

In [4]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS
emulator = emulation.Emulator("3pc.json",mode)

emulator.up()

s_logits = emulator.seal(logits)

[2026-04-08 20:37:50,286]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-08 20:37:50,932] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-08 20:37:50,932] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-08 20:37:50,933] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-08 20:37:50,950] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-08 20:37:50,954] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-08 20:37:52,383] [ForkServerProcess-1] Run : builtin_spu_init at node:0
[2026-04-08 20:37:52,383] [ForkServerProcess-3] Run : builtin_spu_init at node:2
[2026-04-08 20:37:52,383] [ForkServerProcess-2] Run : builtin_spu_init at node:1
I0408 20:37:52.408688 14539     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61931.
W0408 20:37:52.408709 14539     0 external/brpc~/src/brpc/server.cpp:120

### 4.1 baseline

In [5]:
next_id = emulator.run(sampler_baseline)(s_logits)

[2026-04-08 20:37:52,488] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 20:37:52,518] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 20:37:52,582] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-08 20:37:52,582] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-08 20:37:52,582] [ForkServerProcess-3] Run : builtin_spu_run at node:2


[2026-04-08 20:37:52.493] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-08 20:37:52.596] [info] [api.cc:172] [Profiling] SPU execution sampler_baseline completed, input processing took 7.6e-07s, execution took 0.003036441s, output processing took 1.61e-06s, total time 0.003038811s.
[2026-04-08 20:37:52.596] [info] [api.cc:220] HLO profiling: total time 0.000598134
[2026-04-08 20:37:52.596] [info] [api.cc:223] - pphlo.constant, executed 1 times, duration 0.000598134s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-08 20:37:52.596] [info] [api.cc:220] HAL profiling: total time 0
[2026-04-08 20:37:52.596] [info] [api.cc:220] MPC profiling: total time 0
[2026-04-08 20:37:52.596] [info] [api.cc:233] Link details: total send bytes 0, recv bytes 0, send actions 0, recv actions 0


[2026-04-08 20:37:52,584] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:52,586] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:52,591] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:52,603] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 20:37:52,604] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-08 20:37:52,606] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-08 20:37:52,607] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [6]:
print(prompt, tokenizer.decode(next_id), sep="")

python is<|endoftext|>


### 4.2 greedy

In [7]:
next_id = emulator.run(sampler_greedy)(s_logits)

[2026-04-08 20:37:52,626] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 20:37:52,639] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 20:37:52,662] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-08 20:37:52,662] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-08 20:37:52,662] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-08 20:37:52,663] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-08 20:37:52.670] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-08 20:37:52.672] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-08 20:37:52.674] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-08 20:37:52,664] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:52,667] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-08 20:37:53.199] [info] [api.cc:172] [Profiling] SPU execution sampler_greedy completed, input processing took 8.7e-07s, execution took 0.530223238s, output processing took 1.43e-06s, total time 0.530225538s.
[2026-04-08 20:37:53.199] [info] [api.cc:220] HLO profiling: total time 0.5285362640000001
[2026-04-08 20:37:53.199] [info] [api.cc:223] - pphlo.reduce, executed 1 times, duration 0.520719739s, send bytes 21887449 recv bytes 29408329, send actions 844, recv actions 854
[2026-04-08 20:37:53.199] [info] [api.cc:223] - pphlo.iota, executed 1 times, duration 0.005602326s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-08 20:37:53.199] [info] [api.cc:223] - pphlo.convert, executed 3 times, duration 0.001320787s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-08 20:37:53.199] [info] [api.cc:223] - pphlo.free, executed 7 times, duration 0.000841862s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-08 20:37:53.199] [info]

[2026-04-08 20:37:53,201] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 20:37:53,203] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-08 20:37:53,204] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-08 20:37:53,205] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [8]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 4.3 top k

In [9]:
next_id = emulator.run(sampler_top_k)(s_logits)

[2026-04-08 20:37:53,249] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 20:37:53,259] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 20:37:53,341] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-08 20:37:53,341] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-08 20:37:53,341] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-08 20:37:53,342] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:53,343] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:53,345] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-08 20:37:54.292] [info] [api.cc:172] [Profiling] SPU execution sampler_top_k completed, input processing took 9.7e-07s, execution took 0.944405988s, output processing took 1.96e-06s, total time 0.944408918s.
[2026-04-08 20:37:54.293] [info] [api.cc:220] HLO profiling: total time 0.942237707
[2026-04-08 20:37:54.293] [info] [api.cc:223] - pphlo.reduce, executed 1 times, duration 0.479119378s, send bytes 21257241 recv bytes 30647769, send actions 849, recv actions 839
[2026-04-08 20:37:54.293] [info] [api.cc:223] - pphlo.custom_call: mhlo.topk, executed 1 times, duration 0.262509454s, send bytes 9013134 recv bytes 8208654, send actions 141, recv actions 140
[2026-04-08 20:37:54.293] [info] [api.cc:223] - pphlo.less, executed 2 times, duration 0.10017861s, send bytes 3419040 recv bytes 3419040, send actions 9, recv actions 9
[2026-04-08 20:37:54.293] [info] [api.cc:223] - pphlo.log, executed 2 times, duration 0.020589852s, send bytes 0 recv bytes 0, send actions 0, recv actions 0

[2026-04-08 20:37:54,298] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 20:37:54,299] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-08 20:37:54,301] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-08 20:37:54,302] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [10]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 4.4 min p

In [11]:
next_id = emulator.run(sampler_min_p)(s_logits)

[2026-04-08 20:37:54,350] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 20:37:54,361] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 20:37:54,482] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-08 20:37:54,482] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-08 20:37:54,482] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-08 20:37:54,483] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:54,484] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:54,486] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-08 20:37:55.309] [info] [api.cc:172] [Profiling] SPU execution sampler_min_p completed, input processing took 7.6e-07s, execution took 0.814005668s, output processing took 1.82e-06s, total time 0.814008248s.
[2026-04-08 20:37:55.309] [info] [api.cc:220] HLO profiling: total time 0.8106439929999999
[2026-04-08 20:37:55.309] [info] [api.cc:223] - pphlo.reduce, executed 2 times, duration 0.607576507s, send bytes 27766085 recv bytes 30746165, send actions 1085, recv actions 1045
[2026-04-08 20:37:55.309] [info] [api.cc:223] - pphlo.less, executed 2 times, duration 0.099052061s, send bytes 3419040 recv bytes 3419040, send actions 9, recv actions 9
[2026-04-08 20:37:55.309] [info] [api.cc:223] - pphlo.select, executed 2 times, duration 0.023581685s, send bytes 2413440 recv bytes 804480, send actions 3, recv actions 1
[2026-04-08 20:37:55.309] [info] [api.cc:223] - pphlo.log, executed 2 times, duration 0.020404254s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-0

[2026-04-08 20:37:55,312] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 20:37:55,314] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-08 20:37:55,315] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-08 20:37:55,316] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [12]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 4.5 legacy

In [13]:
next_id = emulator.run(sampler_legacy)(s_logits)

[2026-04-08 20:37:55,362] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 20:37:55,373] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 20:37:55,496] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-08 20:37:55,496] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-08 20:37:55,496] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-08 20:37:55,497] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:55,498] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 20:37:55,501] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-08 20:37:59.311] [info] [api.cc:172] [Profiling] SPU execution sampler_legacy completed, input processing took 7.9e-07s, execution took 3.802708104s, output processing took 2.2e-06s, total time 3.802711094s.
[2026-04-08 20:37:59.311] [info] [api.cc:220] HLO profiling: total time 3.798931112
[2026-04-08 20:37:59.311] [info] [api.cc:223] - pphlo.log, executed 3 times, duration 1.754272427s, send bytes 104984640 recv bytes 104984640, send actions 41, recv actions 42
[2026-04-08 20:37:59.311] [info] [api.cc:223] - pphlo.exponential, executed 1 times, duration 0.627770473s, send bytes 60939360 recv bytes 55911360, send actions 75, recv actions 70
[2026-04-08 20:37:59.311] [info] [api.cc:223] - pphlo.reduce, executed 3 times, duration 0.553451666s, send bytes 32526757 recv bytes 26064101, send actions 1084, recv actions 1050
[2026-04-08 20:37:59.311] [info] [api.cc:223] - pphlo.divide, executed 1 times, duration 0.493107814s, send bytes 53296800 recv bytes 55710240, send actions 62,

[2026-04-08 20:37:59,314] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 20:37:59,315] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-08 20:37:59,316] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-08 20:37:59,317] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [14]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


## 5. cleanup

In [15]:
emulator.down()

[2026-04-08 20:37:59,371]-[INFO]-[emulation.py:120]: Shutdown multiprocess cluster...
